In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
from skimage import io, transform
from tqdm import tqdm

# Imports Grounding DINO
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules

# Imports MedSAM
from MedSAM.segment_anything import sam_model_registry
from MedSAM.MedSAM_Inference import medsam_inference

# ==========================================
# FONCTIONS UTILITAIRES
# ==========================================
def calculate_dice(mask_true, mask_pred):
    m_true = np.asarray(mask_true).astype(bool)
    m_pred = np.asarray(mask_pred).astype(bool)
    if m_true.sum() + m_pred.sum() == 0:
        return 1.0
    intersection = np.logical_and(m_true, m_pred).sum()
    return 2 * intersection / (m_true.sum() + m_pred.sum())

# ==========================================
# 1. INITIALISATION DES MODELES
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Appareil detecte : {device}")

register_all_modules()

# Assure-toi que le chemin vers le checkpoint correspond bien à ton entraînement V4
dino_config = 'tumor_config_v4.py'
dino_checkpoint = 'work_dirs/tumor_config_v4/best_coco_bbox_mAP_epoch_14.pth'
dino_model = init_detector(dino_config, dino_checkpoint, device=device)

medsam_checkpoint = "MedSAM/work_dir/MedSAM/medsam_vit_b.pth"
medsam_model = sam_model_registry["vit_b"](checkpoint=medsam_checkpoint)
medsam_model = medsam_model.to(device)
medsam_model.eval()

# ==========================================
# 2. CONFIGURATION DES CHEMINS (MSD Pancreas)
# ==========================================
base_dir = "data/MSD_pancreas"
test_json_path = os.path.join(base_dir, "test.json")
output_folder = "data/outputs_medsam_dino_msd"

os.makedirs(output_folder, exist_ok=True)
os.makedirs("data/results", exist_ok=True)

with open(test_json_path, 'r') as f:
    test_data = json.load(f)

test_images_list = test_data['images'] 

dice_list = []

print(f"\nDebut de l'evaluation sur les {len(test_images_list)} images du fichier test.json...")

# ==========================================
# 3. BOUCLE D'INFERENCE
# ==========================================
for img_info in tqdm(test_images_list, desc="Inference Test Set"):
    file_name = img_info['file_name']
    img_id = img_info['id']
    
    img_path = os.path.join(base_dir, file_name)
    
    mask_rel_path = file_name.replace("/images/", "/masks/")
    mask_path = os.path.join(base_dir, mask_rel_path)

    if not os.path.exists(img_path) or not os.path.exists(mask_path):
        continue
        
    img_np = io.imread(img_path)
    if len(img_np.shape) == 2:
        img_3c = np.repeat(img_np[:, :, None], 3, axis=-1)
    else:
        img_3c = img_np
    H, W, _ = img_3c.shape

    true_seg_raw = io.imread(mask_path)
    true_seg = np.zeros_like(true_seg_raw, dtype=np.uint8)
    true_seg[true_seg_raw == 2] = 1

    has_tumor = true_seg.sum() > 0

    full_medsam_seg = np.zeros((H, W), dtype=np.uint8)
    best_score = 0.0  

    with torch.no_grad():
        # MODIFICATION MAJEURE ICI : On interroge le modele avec le nouveau label specifique
        # Le point final est important pour la tokenization de DINO
        result = inference_detector(dino_model, img_path, text_prompt="pancreatic tumor.")
        pred_instances = result.pred_instances
        
        scores = pred_instances.scores.cpu().numpy()
        bboxes = pred_instances.bboxes.cpu().numpy()
        
        if len(scores) > 0:
            best_score = float(scores.max())
        
        # On garde un seuil bas pour l'etude de la distribution des scores
        mask_conf = scores > 0.1
        valid_boxes = bboxes[mask_conf]
        valid_scores = scores[mask_conf]

        if len(valid_boxes) > 0:
            best_idx = np.argmax(valid_scores)
            best_box = valid_boxes[best_idx]
            
            img_1024 = transform.resize(img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
            img_1024 = (img_1024 - img_1024.min()) / np.clip(img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None)
            img_1024_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
            
            image_embedding = medsam_model.image_encoder(img_1024_tensor)
            
            box_np = np.array([best_box]) 
            box_1024 = box_np / np.array([W, H, W, H]) * 1024
            
            medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)
            full_medsam_seg[medsam_seg > 0] = 1

    dice_score = calculate_dice(true_seg, full_medsam_seg)
    dice_list.append({
        "image_id": img_id,
        "file_name": os.path.basename(file_name),
        "dice": dice_score,
        "has_tumor": has_tumor,
        "best_dino_score": best_score 
    })

# ==========================================
# 4. ANALYSE DES RESULTATS ET DES SEUILS
# ==========================================
df = pd.DataFrame(dice_list)
df.to_csv("data/results/dice_final_report_msd_oneshot.csv", index=False)

df_tumor = df[df['has_tumor'] == True]
df_no_tumor = df[df['has_tumor'] == False]

print("\n" + "="*40)
print("RESULTATS GLOBAUX (Toutes les images)")
print("-" * 40)
print(f"DICE MOYEN : {df['dice'].mean():.4f}")
print(f"DICE MEDIAN : {df['dice'].median():.4f}")

print("\n" + "="*40)
print(f"SCANS AVEC TUMEUR ({len(df_tumor)} images)")
print("-" * 40)
if not df_tumor.empty:
    print(f"DICE MOYEN : {df_tumor['dice'].mean():.4f}")
    print(f"DICE MEDIAN : {df_tumor['dice'].median():.4f}")

print("\n" + "="*40)
print(f"SCANS SANS TUMEUR ({len(df_no_tumor)} images)")
print("-" * 40)
if not df_no_tumor.empty:
    print(f"DICE MOYEN : {df_no_tumor['dice'].mean():.4f}")
    print(f"FAUX POSITIFS (DICE = 0.0) : {len(df_no_tumor[df_no_tumor['dice'] == 0.0])} images")

print("\n" + "="*40)
print("ETUDE DU SEUIL DE DETECTION (DINO SCORE)")
print("-" * 40)

stats = df.groupby('has_tumor')['best_dino_score'].describe()
print("Statistiques des scores de confiance :")
print(stats[['mean', 'min', '50%', 'max']].to_string())

if not df_no_tumor.empty and not df_tumor.empty:
    max_fp_score = df_no_tumor['best_dino_score'].max()
    print(f"\n-> Le score maximal sur un scan SAIN est de : {max_fp_score:.4f}")
    print(f"-> Un seuil strict devrait etre regle a : > {max_fp_score:.4f}")
    
    sim_fp = len(df_no_tumor[df_no_tumor['best_dino_score'] > max_fp_score])
    sim_fn = len(df_tumor[df_tumor['best_dino_score'] <= max_fp_score])
    
    print(f"\nImpact d'un threshold a {max_fp_score:.4f} :")
    print(f"- Faux positifs restants : {sim_fp} / {len(df_no_tumor)}")
    print(f"- Vraies tumeurs ignorees (Faux negatifs) : {sim_fn} / {len(df_tumor)}")
    
    median_tumor_score = df_tumor['best_dino_score'].median()
    recommended_threshold = (max_fp_score + median_tumor_score) / 2 if max_fp_score < median_tumor_score else max_fp_score * 0.9
    
    print(f"\nSi on perd trop de vraies tumeurs, essayer le threshold de compromis : {recommended_threshold:.4f}")
print("="*40)

Appareil detecte : cuda
Loads checkpoint by local backend from path: work_dirs/tumor_config_v4/best_coco_bbox_mAP_epoch_14.pth


/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filena


Debut de l'evaluation sur les 100 images du fichier test.json...


Inference Test Set:   0%|          | 0/100 [00:00<?, ?it/s][nltk_data] Downloading package punkt to ~/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


noun_phrases: ['pancreatic tumor']


...

noun_phrases: ['pancreatic tumor']


Inference Test Set:  99%|█████████▉| 99/100 [02:03<00:01,  1.28s/it][nltk_data] Downloading package punkt to ~/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmcv/cnn/bricks/transformer.py:524: UserWarning: position encoding of key ismissing in MultiheadAttention.
  warnings.warn(f'position encoding of key is'


noun_phrases: ['pancreatic tumor']


Inference Test Set: 100%|██████████| 100/100 [02:03<00:00,  1.23s/it]


RESULTATS GLOBAUX (Toutes les images)
----------------------------------------
DICE MOYEN : 0.2114
DICE MEDIAN : 0.0000

SCANS AVEC TUMEUR (60 images)
----------------------------------------
DICE MOYEN : 0.3524
DICE MEDIAN : 0.3510

SCANS SANS TUMEUR (40 images)
----------------------------------------
DICE MOYEN : 0.0000
FAUX POSITIFS (DICE = 0.0) : 40 images

ETUDE DU SEUIL DE DETECTION (DINO SCORE)
----------------------------------------
Statistiques des scores de confiance :
               mean       min       50%       max
has_tumor                                        
False      0.934892  0.359135  0.965007  0.980581
True       0.931709  0.432952  0.961365  0.983244

-> Le score maximal sur un scan SAIN est de : 0.9806
-> Un seuil strict devrait etre regle a : > 0.9806

Impact d'un threshold a 0.9806 :
- Faux positifs restants : 0 / 40
- Vraies tumeurs ignorees (Faux negatifs) : 59 / 60

Si on perd trop de vraies tumeurs, essayer le threshold de compromis : 0.8825
